# Experiment 1 — Does distant-supervision data augmentation help BABE news-bias detection?

This notebook performs **only the missing experiment**. It does not repeat TF-IDF baselines.

**Comparison**
- A: RoBERTa trained on BABE training sentences only.
- B: the same RoBERTa setup trained on BABE plus a controlled sample from MBIB's Wikipedia-derived sources.
- Both models use the same BABE validation and test examples. Select checkpoints using BABE validation macro-F1.

**Important:** Wikipedia neutrality / NPOV labels are not identical to expert judgments of news bias. This is an experiment in transfer from distant supervision, not a claim that these labels are interchangeable. The final result may improve, stay the same, or get worse.

**Run in Colab with a T4 GPU.** Upload `babe_cleaned.csv` if you do not already have it in the runtime.

## 1. Install and import

In [ ]:
%pip install -q transformers datasets accelerate scikit-learn pandas==2.2.3

In [6]:
import os, random, json, re
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding,
                          EarlyStoppingCallback)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))


Torch: 2.11.0+cu130 | CUDA: True
Tesla T4


## 2. Reuse the existing BABE split when available

In [3]:
DATA_PATH = Path("/content/babe_cleaned.csv")
if not DATA_PATH.exists(): DATA_PATH = Path("babe_cleaned.csv")
assert DATA_PATH.exists(), "Upload babe_cleaned.csv."

# Prefer the exact split files saved by roberta.ipynb.
SPLIT_DIR = Path("/content/babe_sg2_splits")
if not (SPLIT_DIR / "train.csv").exists():
    SPLIT_DIR = Path("babe_sg2_splits")

def normalize_label(value):
    if pd.isna(value): return np.nan
    if isinstance(value, (int, np.integer, float, np.floating)) and value in (0, 1): return int(value)
    s = str(value).strip().lower().replace("_", "-")
    return {"non-biased":0, "non biased":0, "nonbiased":0, "unbiased":0, "0":0,
            "biased":1, "1":1}.get(s, np.nan)

if all((SPLIT_DIR / f"{s}.csv").exists() for s in ["train","validation","test"]):
    train_df = pd.read_csv(SPLIT_DIR / "train.csv")
    val_df = pd.read_csv(SPLIT_DIR / "validation.csv")
    test_df = pd.read_csv(SPLIT_DIR / "test.csv")
    for part in [train_df, val_df, test_df]:
        if "label" not in part.columns: part["label"] = part["label_bias"].map(normalize_label)
        part["label"] = part["label"].astype(int)
    print("Loaded saved split files from", SPLIT_DIR)
else:
    print("Saved split files not found; reproducing the deterministic split from roberta.ipynb.")
    df = pd.read_csv(DATA_PATH)
    df["label"] = df["label_bias"].map(normalize_label)
    df["text"] = df["text"].astype("string").str.strip()
    df = df.dropna(subset=["text","news_link","label"]).copy()
    df = df[df["text"].str.len().gt(0)].copy()
    df["label"] = df["label"].astype(int)
    df["news_link"] = df["news_link"].astype(str).str.strip()
    df = df[~df["news_link"].isin(["","nan","None"])].copy()
    conflict = df.groupby("text")["label"].nunique()
    df = df[~df["text"].isin(conflict[conflict > 1].index)].drop_duplicates("text").reset_index(drop=True)
    df["article_group"] = df["news_link"]
    s1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
    trv_i, te_i = next(s1.split(df, y=df.label, groups=df.article_group))
    trv, test_df = df.iloc[trv_i].copy(), df.iloc[te_i].copy()
    s2 = GroupShuffleSplit(n_splits=1, test_size=0.17647, random_state=43)
    tr_i, va_i = next(s2.split(trv, y=trv.label, groups=trv.article_group))
    train_df, val_df = trv.iloc[tr_i].copy(), trv.iloc[va_i].copy()
    for part in [train_df,val_df,test_df]:
        part.drop(columns=["article_group"], errors="ignore", inplace=True)
    SPLIT_DIR.mkdir(parents=True, exist_ok=True)
    for name,part in [("train",train_df),("validation",val_df),("test",test_df)]:
        part.to_csv(SPLIT_DIR/f"{name}.csv", index=False)

for name,part in [("train",train_df),("validation",val_df),("test",test_df)]:
    print(name, len(part), "labels", part.label.value_counts().sort_index().to_dict(),
          "articles", part.news_link.nunique() if "news_link" in part else "n/a")
assert set(train_df.news_link).isdisjoint(set(val_df.news_link))
assert set(train_df.news_link).isdisjoint(set(test_df.news_link))
assert set(val_df.news_link).isdisjoint(set(test_df.news_link))


Loaded saved split files from /content/babe_sg2_splits
train 2579 labels {0: 1327, 1: 1252} articles 1465
validation 550 labels {0: 266, 1: 284} articles 315
test 508 labels {0: 252, 1: 256} articles 315


## 3. Load and audit the Wikipedia-derived MBIB examples

In [4]:
from datasets import load_dataset

ds = load_dataset(
    "mediabiasgroup/mbib-base",
    split="linguistic_bias"
)

mbib = ds.to_pandas()

print("Rows:", len(mbib))
print("Columns:", mbib.columns.tolist())
print(mbib["dataset_id"].value_counts())

README.md:   0%|          | 0.00/4.22k [00:00<?, ?B/s]

cognitive-bias.csv:   0%|          | 0.00/851k [00:00<?, ?B/s]

fake-news.csv:   0%|          | 0.00/1.00M [00:00<?, ?B/s]

gender-bias.csv:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

mbib-aggregated/hate-speech.csv: reconstructing file:   0%|          |  0.00B / 99.0MB            

...

Generating linguistic_bias split:   0%|          | 0/401862 [00:00<?, ? examples/s]

Generating political_bias split:   0%|          | 0/17704 [00:00<?, ? examples/s]

Generating racial_bias split:   0%|          | 0/9788 [00:00<?, ? examples/s]

Generating text_level_bias split:   0%|          | 0/9018 [00:00<?, ? examples/s]

Rows: 401862
Columns: ['id', 'text', 'label', 'dataset_id']
dataset_id
26    339061
35     34573
3      11730
75     10854
10      3480
9       1444
38       720
Name: count, dtype: int64


## 4. Create a conservative, balanced augmentation sample

In [7]:
# Use only the two Wikipedia-derived sources identified in the MBIB repository:
# 3 = WikiNPOV; 26 = WikiNeutralityCorpus.
# We sample a modest amount to keep training feasible on a T4.
# This is an explicit experimental choice, not an assumption that these labels are perfect.

wiki = mbib[mbib["dataset_id"].astype(str).isin(["3","26"])].copy()
wiki["text"] = wiki["text"].astype(str).str.strip()
wiki["label"] = pd.to_numeric(wiki["label"], errors="coerce")
wiki = wiki.dropna(subset=["text","label"])
wiki = wiki[wiki["label"].isin([0,1])].copy()
wiki["label"] = wiki["label"].astype(int)
wiki = wiki[wiki["text"].str.len().between(20, 1500)].copy()
wiki["text_norm"] = wiki["text"].str.lower().str.replace(r"\s+"," ",regex=True).str.strip()
wiki = wiki.drop_duplicates("text_norm")

# Prevent exact text overlap with any BABE split.
babe_texts = set(pd.concat([train_df.text,val_df.text,test_df.text]).astype(str)
                 .str.lower().str.replace(r"\s+"," ",regex=True).str.strip())
wiki = wiki[~wiki["text_norm"].isin(babe_texts)]
print("Wikipedia-derived rows after cleaning and BABE exact-text dedup:", len(wiki))
display(pd.crosstab(wiki["dataset_id"], wiki["label"]))

# Balanced sample per source/label where possible, up to 2,000 rows per label in total.
# Keep source ID for audit, then remove it from model input.
MAX_PER_LABEL = 2000
sample_parts=[]
for label in [0,1]:
    part = wiki[wiki.label == label]
    take = min(MAX_PER_LABEL, len(part))
    sample_parts.append(part.sample(n=take, random_state=SEED + label))
wiki_sample = pd.concat(sample_parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
print("Augmentation sample:", len(wiki_sample))
print("Label counts:", wiki_sample.label.value_counts().sort_index().to_dict())
display(wiki_sample.groupby(["dataset_id","label"]).size().rename("n").reset_index())


Wikipedia-derived rows after cleaning and BABE exact-text dedup: 349834


label,0,1
dataset_id,,
3,10034,1498
26,178036,160266


Augmentation sample: 4000
Label counts: {0: 2000, 1: 2000}


,dataset_id,label,n
0,3,0,103
1,3,1,18
2,26,0,1897
3,26,1,1982


## 5. Train both conditions with the same model settings

In [8]:
MODEL_NAME = "FacebookAI/roberta-base"
MAX_LENGTH = 256
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def to_hf(frame):
    d = Dataset.from_pandas(frame[["text","label"]].copy().reset_index(drop=True), preserve_index=False)
    def tok(batch): return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)
    return d.map(tok, batched=True, remove_columns=["text"])

train_babe_hf = to_hf(train_df)
train_aug_hf = to_hf(pd.concat([
    train_df[["text","label"]],
    wiki_sample[["text","label"]]
], ignore_index=True))
val_hf = to_hf(val_df)
test_hf = to_hf(test_df)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels,pred),
            "macro_f1": f1_score(labels,pred,average="macro",zero_division=0)}

def train_condition(condition, train_hf):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, id2label={0:"Non-biased",1:"Biased"},
        label2id={"Non-biased":0,"Biased":1})
    outdir = f"/content/augmentation_experiment_{condition}"
    args = TrainingArguments(
        output_dir=outdir, learning_rate=1e-5,
        per_device_train_batch_size=8, per_device_eval_batch_size=16,
        num_train_epochs=5, weight_decay=0.01,
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="eval_macro_f1",
        greater_is_better=True, fp16=torch.cuda.is_available(),
        logging_steps=50, report_to="none", seed=SEED, data_seed=SEED,
        save_total_limit=1)
    trainer = Trainer(model=model,args=args,train_dataset=train_hf,
        eval_dataset=val_hf,data_collator=collator,compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)])
    trainer.train()
    val_metrics = trainer.evaluate(val_hf)
    # Test is evaluated only after checkpoint selection is complete.
    test_out = trainer.predict(test_hf)
    pred = np.argmax(test_out.predictions, axis=-1)
    result = {
        "condition":condition,
        "best_validation_macro_f1":float(trainer.state.best_metric),
        "test_accuracy":float(accuracy_score(test_out.label_ids,pred)),
        "test_macro_f1":float(f1_score(test_out.label_ids,pred,average="macro",zero_division=0)),
        "test_biased_f1":float(f1_score(test_out.label_ids,pred,pos_label=1,zero_division=0)),
        "test_nonbiased_f1":float(f1_score(test_out.label_ids,pred,pos_label=0,zero_division=0)),
        "test_predictions":pred.tolist()
    }
    trainer.save_model(outdir + "/best_model")
    return result, trainer

# Two training runs; on a T4 this can take a while.
baseline_result, baseline_trainer = train_condition("babe_only", train_babe_hf)
augmented_result, augmented_trainer = train_condition("babe_plus_wikipedia", train_aug_hf)


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.421060,0.433130,3,0.814545,0.814484


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 6. Compare results and inspect error types

In [9]:
results = pd.DataFrame([
    {k:v for k,v in baseline_result.items() if k != "test_predictions"},
    {k:v for k,v in augmented_result.items() if k != "test_predictions"}
])
display(results)
print("Change in test macro-F1 (augmented - BABE only):",
      round(augmented_result["test_macro_f1"] - baseline_result["test_macro_f1"], 4))

y_test = test_df["label"].to_numpy()
for result in [baseline_result, augmented_result]:
    print("\\n###", result["condition"])
    print(classification_report(y_test, result["test_predictions"],
          labels=[0,1], target_names=["Non-biased","Biased"], digits=4, zero_division=0))
    print("Confusion matrix:\\n", confusion_matrix(y_test,result["test_predictions"]))


,condition,best_validation_macro_f1,test_accuracy,test_macro_f1,test_biased_f1,test_nonbiased_f1
0,babe_only,0.825445,0.818898,0.818760,0.813765,0.823755
1,babe_plus_wikipedia,0.814484,0.820866,0.820849,0.819085,0.822612


Change in test macro-F1 (augmented - BABE only): 0.0021
\n### babe_only
              precision    recall  f1-score   support

  Non-biased     0.7963    0.8532    0.8238       252
      Biased     0.8445    0.7852    0.8138       256

    accuracy                         0.8189       508
   macro avg     0.8204    0.8192    0.8188       508
weighted avg     0.8206    0.8189    0.8187       508

Confusion matrix:\n [[215  37]
 [ 55 201]]
\n### babe_plus_wikipedia
              precision    recall  f1-score   support

  Non-biased     0.8084    0.8373    0.8226       252
      Biased     0.8340    0.8047    0.8191       256

    accuracy                         0.8209       508
   macro avg     0.8212    0.8210    0.8208       508
weighted avg     0.8213    0.8209    0.8208       508

Confusion matrix:\n [[211  41]
 [ 50 206]]


## 7. Save the experiment artefacts

In [10]:
OUT = Path("/content/newsbias_augmentation_results")
OUT.mkdir(parents=True, exist_ok=True)
results.to_csv(OUT/"comparison_metrics.csv", index=False)
preds = test_df[["text","label","news_link"]].reset_index(drop=True).copy()
preds["pred_babe_only"] = baseline_result["test_predictions"]
preds["pred_babe_plus_wikipedia"] = augmented_result["test_predictions"]
preds["baseline_correct"] = preds.label == preds.pred_babe_only
preds["augmented_correct"] = preds.label == preds.pred_babe_plus_wikipedia
preds.to_csv(OUT/"test_predictions_comparison.csv", index=False)
wiki_sample.drop(columns=["text_norm"],errors="ignore").to_csv(OUT/"wiki_augmentation_sample.csv",index=False)
with open(OUT/"experiment_summary.json","w") as f:
    json.dump({
        "seed":SEED,"model":MODEL_NAME,"max_length":MAX_LENGTH,
        "wiki_sample_size":len(wiki_sample),
        "wiki_source_ids":sorted(wiki_sample.dataset_id.astype(str).unique().tolist()),
        "baseline":{k:v for k,v in baseline_result.items() if k!="test_predictions"},
        "augmented":{k:v for k,v in augmented_result.items() if k!="test_predictions"},
        "caveat":"Wikipedia-derived labels are distant supervision and may not match expert-annotated news bias."
    },f,indent=2)
print("Saved to", OUT)
display(preds[(preds.baseline_correct != preds.augmented_correct)].head(25))


Saved to /content/newsbias_augmentation_results


,text,label,news_link,pred_babe_only,pred_babe_plus_wikipedia,baseline_correct,augmented_correct
2,"A 24-year-old woman in Uganda, unable to trave...",0,https://www.foxnews.com/world/coronavirus-pers...,1,0,False,True
20,A review of his posts in online message boards...,1,https://eu.usatoday.com/story/news/nation/2020...,1,0,True,False
28,Activists planning to line California roadways...,0,https://www.nbcnews.com/tech/internet/anti-vac...,0,1,True,False
44,"Andreyah Garland, a 44-year-old single mother ...",1,https://www.reuters.com/article/domesticNews/i...,0,1,False,True
46,"Antifa, short for “anti-fascist,” is an amorph...",0,https://www.reuters.com/article/uk-factcheck-f...,1,0,False,True
53,As people sip cafecito and eat croquetas at th...,1,https://www.foxnews.com/politics/bernie-sander...,1,0,True,False
73,Biden spoke passionately Saturday night about ...,1,https://www.nbcnews.com/politics/2020-election...,0,1,False,True
83,But Bannon — the former Harvard Business Schoo...,1,https://www.politico.com/story/2017/10/21/stev...,0,1,False,True
89,But instead of closing loopholes that allow cr...,1,https://www.foxnews.com/opinion/dem-gun-bill-t...,1,0,True,False
126,"Ever since, Sanders has portrayed himself as a...",0,https://www.alternet.org/2019/07/stickin-to-hi...,0,1,True,False


## Interpretation checklist for the paper

- Did augmentation improve **validation macro-F1** and final test macro-F1, or only one of them?
- Which class changed most? Did biased recall improve at the cost of non-biased precision?
- Did source augmentation change error patterns?
- Do not claim causality beyond this controlled comparison, and do not claim novelty until related work has been checked.
- This notebook evaluates the same BABE test set for both conditions. If that test set has already been repeatedly used to guide model choices, disclose this limitation or create a new locked article-disjoint test set for a final study.
- SHAP is a follow-up analysis, not the primary novelty claim; the key experiment here is controlled distant-supervision augmentation.

In [1]:
import os

paths = [
    "/content/newsbias_roberta_retrain",
    "/content/newsbias_augmentation_experiment_babe_only",
    "/content/newsbias_augmentation_experiment_babe_plus_wikipedia",
    "/content/babe_sg2_splits",
    "/content/newsbias_augmentation_results",
]

for path in paths:
    print(f"\n{path}")
    if os.path.exists(path):
        print(os.listdir(path)[:20])
    else:
        print("Not found")


/content/newsbias_roberta_retrain
Not found

/content/newsbias_augmentation_experiment_babe_only
Not found

/content/newsbias_augmentation_experiment_babe_plus_wikipedia
Not found

/content/babe_sg2_splits
Not found

/content/newsbias_augmentation_results
Not found


In [2]:
import pandas as pd
from pathlib import Path

split_dir = Path("/content/babe_sg2_splits")

for name in ["train", "validation", "test"]:
    df = pd.read_csv(split_dir / f"{name}.csv")
    print(
        name,
        "| rows:", len(df),
        "| columns:", df.columns.tolist()
    )

train | rows: 2579 | columns: ['text', 'news_link', 'outlet', 'topic', 'type', 'label_bias', 'label_opinion', 'biased_words', 'label']
validation | rows: 550 | columns: ['text', 'news_link', 'outlet', 'topic', 'type', 'label_bias', 'label_opinion', 'biased_words', 'label']
test | rows: 508 | columns: ['text', 'news_link', 'outlet', 'topic', 'type', 'label_bias', 'label_opinion', 'biased_words', 'label']


In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
import os
import shutil

source = "/content"
destination = "/content/drive/MyDrive/NewsBias_Project_Backup"

folders = [
    "augmentation_experiment_babe_only",
    "augmentation_experiment_babe_plus_wikipedia",
    "babe_sg2_splits",
    "newsbias_augmentation_results"
]

os.makedirs(destination, exist_ok=True)

for folder in folders:
    src = os.path.join(source, folder)
    dst = os.path.join(destination, folder)

    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"Saved: {folder}")
    else:
        print(f"NOT FOUND: {folder}")

print("\nBackup location:", destination)

Saved: augmentation_experiment_babe_only
Saved: augmentation_experiment_babe_plus_wikipedia
Saved: babe_sg2_splits
Saved: newsbias_augmentation_results

Backup location: /content/drive/MyDrive/NewsBias_Project_Backup


In [13]:
import os

backup = "/content/drive/MyDrive/NewsBias_Project_Backup"

for folder in os.listdir(backup):
    path = os.path.join(backup, folder)
    print(f"\n{folder}")
    for root, dirs, files in os.walk(path):
        for file in files[:5]:
            print("  ", os.path.relpath(os.path.join(root, file), backup))


augmentation_experiment_babe_only
   augmentation_experiment_babe_only/best_model/model.safetensors
   augmentation_experiment_babe_only/best_model/tokenizer.json
   augmentation_experiment_babe_only/best_model/tokenizer_config.json
   augmentation_experiment_babe_only/best_model/training_args.bin
   augmentation_experiment_babe_only/best_model/config.json
   augmentation_experiment_babe_only/checkpoint-969/scheduler.pt
   augmentation_experiment_babe_only/checkpoint-969/model.safetensors
   augmentation_experiment_babe_only/checkpoint-969/rng_state.pth
   augmentation_experiment_babe_only/checkpoint-969/tokenizer.json
   augmentation_experiment_babe_only/checkpoint-969/tokenizer_config.json

augmentation_experiment_babe_plus_wikipedia
   augmentation_experiment_babe_plus_wikipedia/best_model/model.safetensors
   augmentation_experiment_babe_plus_wikipedia/best_model/tokenizer.json
   augmentation_experiment_babe_plus_wikipedia/best_model/tokenizer_config.json
   augmentation_experime